# Improved CMNet Training on Google Colab
This notebook applies our architectural improvements (ResNet50 backbone, adaptive fusion) and training optimizations (AdamW, Cosine Annealing, Label Smoothing, RandAugment) to the CMNet model.

In [ ]:
!git clone https://github.com/hellloxiaotian/CMNet.git
%cd CMNet

In [ ]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('experiment/visual/confusion_matrix', exist_ok=True)
os.makedirs('experiment/rafdb', exist_ok=True)
os.makedirs('checkpoint', exist_ok=True)
os.makedirs('log', exist_ok=True)

!pip install -U gdown
# Download datasets
!gdown --folder https://drive.google.com/drive/folders/1OjFbQ7ykV5x96MrAMySHwMpaJKzFgGSe -O data_tmp
# Download models
!gdown --folder https://drive.google.com/drive/folders/1kPUbCKoDKesxbpubPTxedD3DWsGGJ1qj -O models_tmp

# Move contents to appropriate folders
!mv data_tmp/* data/ || true
!mv models_tmp/* models/ || true

In [ ]:
%%writefile network/my_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from .mya import myCBAM

import torchvision.models as models
from .attention import CBAM

# 融合
class Model_1(nn.Module):      # 偶数长宽融合+loss
    def __init__(self,num_class=7,device='cpu'):
        super(Model_1,self).__init__()
        self.resnet=models.resnet18()
        self.resnet2=models.resnet18()
        self.attention=myCBAM(512)
        checkpoint=torch.load('models/resnet18_msceleb.pth',map_location=device)
        self.resnet.load_state_dict(checkpoint['state_dict'],strict=True)
        self.features1=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features2=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features3=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features4=nn.Sequential(*list(self.resnet.children())[-3:-2])
        self.fc=nn.Linear(512,num_class)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.relu=nn.ReLU()
    def forward(self,x):
        w=x.size(3)
        w_2=int(w/2)

        x2=x[:,:,:,0:w_2]
        x3=x[:,:,:,w_2:w]
        x1=self.features1(x)
        x2=self.features2(x2)
        x3=self.features3(x3)

        x4=torch.cat([x2,x3],dim=3)

        x2=self.avgpool(x2)
        x2=x2.view(x.size(0),-1)
        x2=torch.unsqueeze(x2,1)
        x3=self.avgpool(x3)
        x3=x3.view(x.size(0),-1)
        x3=torch.unsqueeze(x3,1)

        heads=torch.cat([x2,x3],dim=1)
        heads = F.log_softmax(heads,dim=1)

        x5=x1+x4

        x5=self.features4(x5)
        x5=self.avgpool(x5)
        x5=x5.view(x.size(0),-1)
        x5=self.fc(x5)
        return x5,heads

# 融合+my_CBAM
class Model_2(nn.Module):      # 偶数长宽融合+loss
    def __init__(self,num_class=7,device='cpu'):
        super(Model_2,self).__init__()
        self.resnet=models.resnet18()
        self.resnet2=models.resnet18()
        checkpoint=torch.load('models/resnet18_msceleb.pth',map_location=device)
        self.resnet.load_state_dict(checkpoint['state_dict'],strict=True)
        self.features1=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features2=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features3=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features4=nn.Sequential(*list(self.resnet.children())[-3:-2])
        self.attention=myCBAM(512)
        self.fc=nn.Linear(512,num_class)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.relu=nn.ReLU()
    def forward(self,x):
        w=x.size(3)
        w_2=int(w/2)

        x2=x[:,:,:,0:w_2]
        x3=x[:,:,:,w_2:w]
        x1=self.features1(x)
        x2=self.features2(x2)
        x3=self.features3(x3)

        x4=torch.cat([x2,x3],dim=3)

        x2=self.avgpool(x2)
        x2=x2.view(x.size(0),-1)
        x2=torch.unsqueeze(x2,1)
        x3=self.avgpool(x3)
        x3=x3.view(x.size(0),-1)
        x3=torch.unsqueeze(x3,1)

        heads=torch.cat([x2,x3],dim=1)
        heads = F.log_softmax(heads,dim=1)

        x5=x1+x4

        x5=self.features4(x5)
        x5=self.attention(x5)
        x5=self.avgpool(x5)
        x5=x5.view(x.size(0),-1)
        x5=self.fc(x5)
        return x5,heads
    
# 融合+对称损失
class Model_3(nn.Module):      # 偶数长宽融合+loss
    def __init__(self,num_class=7,device='cpu'):
        super(Model_3,self).__init__()
        self.resnet=models.resnet18()
        self.resnet2=models.resnet18()
        checkpoint=torch.load('models/resnet18_msceleb.pth',map_location=device)
        self.resnet.load_state_dict(checkpoint['state_dict'],strict=True)
        self.features1=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features2=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features3=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features4=nn.Sequential(*list(self.resnet.children())[-3:-2])
        self.fc=nn.Linear(512,num_class)
        self.attention=myCBAM(512)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.relu=nn.ReLU()
    def forward(self,x):
        w=x.size(3)
        w_2=int(w/2)

        x2=x[:,:,:,0:w_2]
        x3=x[:,:,:,w_2:w]
        x1=self.features1(x)
        x2=self.features2(x2)
        x3=self.features3(x3)

        x4=torch.cat([x2,x3],dim=3)

        x2=self.avgpool(x2)
        x2=x2.view(x.size(0),-1)
        x2=torch.unsqueeze(x2,1)
        x3=self.avgpool(x3)
        x3=x3.view(x.size(0),-1)
        x3=torch.unsqueeze(x3,1)

        heads=torch.cat([x2,x3],dim=1)
        heads = F.log_softmax(heads,dim=1)

        x5=x1+x4

        x5=self.features4(x5)
        x5=self.avgpool(x5)
        x5=x5.view(x.size(0),-1)
        x5=self.fc(x5)
        return x5,heads
    
# 融合+CBAM+对称损失
class Model_4(nn.Module):      # 偶数长宽融合+loss
    def __init__(self,num_class=7,device='cpu'):
        super(Model_4,self).__init__()
        self.resnet=models.resnet18()
        checkpoint=torch.load('models/resnet18_msceleb.pth',map_location=device)
        self.resnet.load_state_dict(checkpoint['state_dict'],strict=True)
        self.features1=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features2=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features3=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features4=nn.Sequential(*list(self.resnet.children())[-3:-2])
        self.attention=CBAM(512)
        self.fc=nn.Linear(512,num_class)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.relu=nn.ReLU()
    def forward(self,x):
        w=x.size(3)
        w_2=int(w/2)

        x2=x[:,:,:,0:w_2]
        x3=x[:,:,:,w_2:w]
        x1=self.features1(x)
        x2=self.features2(x2)
        x3=self.features3(x3)

        x4=torch.cat([x2,x3],dim=3)

        x2=self.avgpool(x2)
        x2=x2.view(x.size(0),-1)
        x2=torch.unsqueeze(x2,1)
        x3=self.avgpool(x3)
        x3=x3.view(x.size(0),-1)
        x3=torch.unsqueeze(x3,1)

        heads=torch.cat([x2,x3],dim=1)
        heads = F.log_softmax(heads,dim=1)

        x5=x1+x4

        x5=self.features4(x5)
        x5=self.attention(x5)
        x5=self.avgpool(x5)
        x5=x5.view(x.size(0),-1)
        x5=self.fc(x5)
        return x5,heads
    
# 融合+my_CBAM+对称损失
class Model_5(nn.Module):      # 偶数长宽融合+loss
    def __init__(self,num_class=7,device='cpu'):
        super(Model_5,self).__init__()
        self.resnet=models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        
        self.features1=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features2=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features3=nn.Sequential(*list(self.resnet.children())[:-3])
        self.features4=nn.Sequential(*list(self.resnet.children())[-3:-2])
        
        # Adaptive fusion 1x1 conv to merge x1 and x4
        self.fusion = nn.Conv2d(1024 * 2, 1024, kernel_size=1)
        
        self.attention=myCBAM(2048)
        self.fc=nn.Linear(2048,num_class)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.relu=nn.ReLU()
    def forward(self,x):
        w=x.size(3)
        w_2=int(w/2)

        x2=x[:,:,:,0:w_2]
        x3=x[:,:,:,w_2:w]
        x1=self.features1(x)
        x2=self.features2(x2)
        x3=self.features3(x3)

        x4=torch.cat([x2,x3],dim=3)

        x2_pool=self.avgpool(x2)
        x2_pool=x2_pool.view(x.size(0),-1)
        x2_pool=torch.unsqueeze(x2_pool,1)
        x3_pool=self.avgpool(x3)
        x3_pool=x3_pool.view(x.size(0),-1)
        x3_pool=torch.unsqueeze(x3_pool,1)

        heads=torch.cat([x2_pool,x3_pool],dim=1)
        heads = F.log_softmax(heads,dim=1)

        # Adaptive fusion instead of simple addition
        x5 = torch.cat([x1, x4], dim=1)
        x5 = self.fusion(x5)

        x5=self.features4(x5)
        
        x5=self.attention(x5)

        x5=self.avgpool(x5)
        x5=x5.view(x.size(0),-1)
        x5=self.fc(x5)
        return x5,heads


In [ ]:
%%writefile train_rafdb.py
import os
import time
import shutil
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import datetime

from utils.loss import PartitionLoss
from network.my_model import *
from tqdm import tqdm
plt.switch_backend('agg')
now = datetime.datetime.now()
time_str = now.strftime("[%m-%d]-[%H-%M]-")

dataset_name='rafdb'

data_path               = 'data/'+dataset_name
model_name              = dataset_name+'_'
checkpoint_path         = './checkpoint/' + model_name+time_str +'.pth'
best_checkpoint_path    = './checkpoint/'+model_name+time_str+'_best.pth'
txt_name                = './log/' + model_name +time_str+  '.txt'
curve_name              = './log/' + model_name +time_str+  '.png'

device          = '0'
net             = 11
alpha           = 0.9
eval            = False
lr              = 0.01
momentum        = 0.9
weight_decay    = 1e-4
epochs          = 100
ls              = 15
batch_size      = 32
workers         = 8
print_freq      = 100
pretrained      =False

traindir = os.path.join(data_path, 'train')
valdir = os.path.join(data_path, 'test')

os.environ["CUDA_VISIBLE_DEVICES"] = device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def main():
    best_acc=0
    start_epoch=0
    print('Training time: ' + now.strftime("%m-%d %H:%M"))
    model=Model_5(num_class=7,device=device)
    model=model.to(device)
    criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    criterion_pt = PartitionLoss()      # 最大化注意力方差
    optimizer=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    recorder = RecorderMeter( epochs)
    cudnn.benchmark=True        #加速网络

    train_dataset = datasets.ImageFolder(
        traindir,
        transforms.Compose([
            transforms.Resize((224,224)),
            transforms.RandAugment(),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225]),
            transforms.RandomErasing(scale=(0.02,0.25)),
        ])
    )

    val_dataset=datasets.ImageFolder(
        valdir,
        transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
        ])
    )

    train_loader=torch.utils.data.DataLoader(
        train_dataset,
        batch_size= batch_size,
        shuffle=True,
        num_workers= workers,       #数据加载器数量，不可小于batchsize
        pin_memory=True
    )
 
    val_loader=torch.utils.data.DataLoader(
        val_dataset,
        batch_size= batch_size,
        shuffle=False,
        num_workers= workers,
        pin_memory=True
    )
 
    with open(txt_name,'a') as f:
        f.write('model: '+str(net)+'\n'+'time: '+time_str+'\n')

    for epoch in tqdm(range( start_epoch, epochs)):
        start_time=time.time()
        current_learning_rate=optimizer.state_dict()['param_groups'][0]['lr']
        tqdm.write('Current learning rate: '+str(current_learning_rate))
        with open(txt_name, 'a') as f:
            f.write('Current learning rate: ' + str(current_learning_rate) + '\n')

        train_acc,train_los=train(train_loader,model,criterion_cls,criterion_pt,optimizer,epoch+1)       #返回一个训练epoch平均精度和损失
        val_acc,val_los=validate(val_loader,model,criterion_cls,criterion_pt)        #返回一个验证epoch平均精度和损失
        scheduler.step()

        recorder.update(epoch,train_los,train_acc,val_los,val_acc)      #一个epoch记录精度和损失
        recorder.plot_curve(curve_name)      #绘制图形

        is_best=val_acc>best_acc
        best_acc=max(best_acc,val_acc)

        tqdm.write('Current best accuracy: '+str(best_acc.item()))

        with open(txt_name, 'a') as f:
            f.write('********************Current best accuracy: ' + str(best_acc.item()) + '\n')        #记录验证最高精度

        save_checkpoint({'epoch': epoch + 1,
                         'state_dict': model.state_dict(),
                         'best_acc': best_acc,
                         'optimizer': optimizer.state_dict(),
                         'recorder': recorder,}, is_best)

        end_time = time.time()
        epoch_time = end_time - start_time
        tqdm.write("An Epoch Time: "+ str(epoch_time))
        with open(txt_name, 'a') as f:      #写入一个epoch时间
            f.write('An epoch time: '+str(epoch_time) + '\n')


    # 训练
def train(train_loader,model,criterion_cls,criterion_pt,optimizer,epoch):
    losses = AverageMeter('Loss', ':.4f')
    top1 = AverageMeter('Accuracy', ':6.3f')
    progress = ProgressMeter(len(train_loader),     #传入多少个batch，损失和精度
                             [losses, top1],
                             prefix="Epoch: [{}]".format(epoch))

    model.train()
    #一个epoch
    for i,(images, targets) in enumerate(train_loader):
        targets=targets.to(device)
        images=images.to(device)
        out,heads=model(images)

        loss = alpha*criterion_cls(out,targets) +  (1-alpha)*criterion_pt(heads)   #89.3 89.4
        acc=accuracy(out,targets)
        losses.update(loss.item(),images.size(0))       #传入一个batch平均损失，batch_size，计算平均值
        top1.update(acc.item(),images.size(0))        #传入一个batch平均精度，batch_size，计算平均值

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i% print_freq==0:        #显示训练精度和损失
            progress.display(i)
    
    return top1.avg, losses.avg     #返回一个epoch平均精度和损失

    # 验证
def validate(val_loader,model,criterion_cls,criterion_pt):
    losses = AverageMeter('Loss', ':.4f')
    top1 = AverageMeter('Accuracy', ':6.3f')
    progress = ProgressMeter(len(val_loader),
                             [losses, top1],
                             prefix='Test: ')
    model.eval()
    with torch.no_grad():
        for i,(images, targets) in enumerate(val_loader):

            targets=targets.to(device)
            images=images.to(device)

            out,heads=model(images)
            loss = criterion_cls(out,targets) + criterion_pt(heads)

            acc=accuracy(out,targets)
            losses.update(loss.item(),images.size(0))       #传入一个batch平均损失，batch_size，计算平均值
            top1.update(acc,images.size(0))        #传入一个batch平均精度，batch_size，计算平均值

            if i %  print_freq == 0:
                progress.display(i)
        tqdm.write(' **** Accuracy {top1.avg:.3f} *** '.format(top1=top1))
        # print(' **** Accuracy {top1.avg:.3f} *** '.format(top1=top1))       #输出验证平均精度
        with open(txt_name, 'a') as f:
            f.write(' * Accuracy {top1.avg:.3f}'.format(top1=top1) + '\n')      #写入验证平均精度

    return top1.avg,losses.avg
            
    #保存模型
def save_checkpoint(state, is_best):
    torch.save(state,  checkpoint_path)     #保存模型
    if is_best:     #若是最高精度，保存至最高精度模型
        shutil.copyfile( checkpoint_path,  best_checkpoint_path)

    #计算均值
class AverageMeter(object):     
    """Computes and stores the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

    #显示进度
class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)     #获取格式
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]       #
        entries += [str(meter) for meter in self.meters]
        print_txt = '\t'.join(entries)
        tqdm.write(print_txt)
        # print(print_txt)        #输出格式
        with open(txt_name, 'a') as f:      #写入格式
            f.write(print_txt + '\n')

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

    #返回一个batch精度
def accuracy(logits,labels):
    acc=(logits.argmax(dim=-1)==labels).float().mean()
    return acc*100.0

    #绘图
class RecorderMeter(object):
    """Computes and stores the minimum loss value and its epoch index"""

    def __init__(self, total_epoch):
        self.reset(total_epoch)

    def reset(self, total_epoch):
        self.total_epoch = total_epoch
        self.current_epoch = 0
        self.epoch_losses = np.zeros((self.total_epoch, 2), dtype=np.float32)    # [epoch, train/val]
        self.epoch_accuracy = np.zeros((self.total_epoch, 2), dtype=np.float32)  # [epoch, train/val]

    def update(self, idx, train_loss, train_acc, val_loss, val_acc):
        self.epoch_losses[idx, 0] = train_loss * 30
        self.epoch_losses[idx, 1] = val_loss * 30
        self.epoch_accuracy[idx, 0] = train_acc
        self.epoch_accuracy[idx, 1] = val_acc
        self.current_epoch = idx + 1

    def plot_curve(self, save_path):

        title = 'the accuracy/loss curve of train/val'
        dpi = 80
        width, height = 1800, 800
        legend_fontsize = 10
        figsize = width / float(dpi), height / float(dpi)

        fig = plt.figure(figsize=figsize)
        x_axis = np.array([i for i in range(self.total_epoch)])  # epochs
        y_axis = np.zeros(self.total_epoch)

        plt.xlim(0, self.total_epoch)
        plt.ylim(0, 100)
        interval_y = 5
        interval_x = 5
        plt.xticks(np.arange(0, self.total_epoch + interval_x, interval_x))
        plt.yticks(np.arange(0, 100 + interval_y, interval_y))
        plt.grid()
        plt.title(title, fontsize=20)
        plt.xlabel('the training epoch', fontsize=16)
        plt.ylabel('accuracy', fontsize=16)

        y_axis[:] = self.epoch_accuracy[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle='-', label='train-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_accuracy[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle='-', label='valid-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle=':', label='train-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle=':', label='valid-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        if save_path is not None:
            fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
            tqdm.write('Saved figure')
        plt.close(fig)
class DrawConfusionMatrix:
    def __init__(self, labels_name, normalize=True,path='x'):
        # """
		# normalize：是否设元素为百分比形式
        # """
        self.path=path
        self.normalize = normalize
        self.labels_name = labels_name
        self.num_classes = len(labels_name)
        self.matrix = np.zeros((self.num_classes, self.num_classes), dtype="float32")

    def update(self, predicts, labels):
        # """

        # :param predicts: 一维预测向量，eg：array([0,5,1,6,3,...],dtype=int64)
        # :param labels:   一维标签向量：eg：array([0,5,0,6,2,...],dtype=int64)
        # :return:
        # """
        for predict, label in zip(predicts, labels):
            self.matrix[predict, label] += 1

    def getMatrix(self,normalize=True):
        # """
        # 根据传入的normalize判断要进行percent的转换，
        # 如果normalize为True，则矩阵元素转换为百分比形式，
        # 如果normalize为False，则矩阵元素就为数量
        # Returns:返回一个以百分比或者数量为元素的矩阵

        # """
        if normalize:
            per_sum = self.matrix.sum(axis=1)  # 计算每行的和，用于百分比计算
            for i in range(self.num_classes):
                self.matrix[i] =(self.matrix[i] / per_sum[i])   # 百分比转换
            self.matrix=np.around(self.matrix*100, 2)   # 保留2位小数点

            self.matrix[np.isnan(self.matrix)] = 0  # 可能存在NaN，将其设为0
        return self.matrix

    def drawMatrix(self):
        self.matrix = self.getMatrix(self.normalize)
        plt.imshow(self.matrix, cmap=plt.cm.Blues)  # 仅画出颜色格子，没有值
        # plt.title("Normalized confusion matrix")  # title
        plt.xlabel("Predict label")
        plt.ylabel("Truth label")
        plt.yticks(range(self.num_classes), self.labels_name)  # y轴标签
        plt.xticks(range(self.num_classes), self.labels_name, rotation=45)  # x轴标签

        for x in range(self.num_classes):
            for y in range(self.num_classes):
                value = float(format('%.2f' % self.matrix[y, x]))  # 数值处理
                plt.text(x, y, value, verticalalignment='center', horizontalalignment='center')  # 写值

        plt.tight_layout()  # 自动调整子图参数，使之填充整个图像区域

        # plt.colorbar()  # 色条
        plt.savefig('experiment/visual/confusion_matrix/'+self.path+'.png', bbox_inches='tight')  # bbox_inches='tight'可确保标签信息显示全
        plt.show()

if __name__ == '__main__':
    main()


In [ ]:
%%writefile train_affectnet-7.py
import os
import time
import shutil
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torch.utils.data.distributed
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import datetime

from utils.loss import PartitionLoss,ImbalancedDatasetSampler
from network.my_model import *
from tqdm import tqdm
plt.switch_backend('agg')
now = datetime.datetime.now()
time_str = now.strftime("[%m-%d]-[%H-%M]-")

dataset_name='affectnet-7'

data_path               = 'data/'+dataset_name
model_name              = dataset_name+'_'
checkpoint_path         = './checkpoint/' + model_name+time_str +'.pth'
best_checkpoint_path    = './checkpoint/'+model_name+time_str+'_best.pth'
txt_name                = './log/' + model_name +time_str+  '.txt'
curve_name              = './log/' + model_name +time_str+  '.png'

device          = 'cpu'
net             = 11
alpha           = 0.9
eval            = False
lr              = 0.0001
momentum        = 0.9
weight_decay    = 1e-4
epochs          = 40
ls              = 15
batch_size      = 256
workers         = 8
print_freq      = 1000
pretrained      =False

traindir = os.path.join(data_path, 'train')
valdir = os.path.join(data_path, 'test')

def main():

    best_acc=0
    start_epoch=0
    print('Training time: ' + now.strftime("%m-%d %H:%M"))
    
    model = Model_5(num_class=7,device=device)
    model=model.to(device)
    params = list(model.parameters())
    criterion_cls = nn.CrossEntropyLoss(label_smoothing=0.1).to(device)
    criterion_pt = PartitionLoss()      # 最大化注意力方差
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    recorder = RecorderMeter( epochs)
    cudnn.benchmark=True        #加速网络

    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandAugment(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        transforms.RandomErasing(),
        ])
    
    train_dataset = datasets.ImageFolder(f'{data_path}/train', transform = data_transforms)   # loading statically

    print('Whole train set size:', train_dataset.__len__())

    train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size = batch_size,
                                               num_workers = workers,
                                               sampler=ImbalancedDatasetSampler(train_dataset),
                                               shuffle = False, 
                                               pin_memory = True)


    data_transforms_val = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])])    
      
    val_dataset = datasets.ImageFolder(f'{data_path}/val', transform = data_transforms_val)    # loading statically

 
    val_loader=torch.utils.data.DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=workers,
        pin_memory=True
    )
    print('Validation set size:', val_dataset.__len__())
 
    if  eval:
        validate(val_loader, model, criterion_cls)
        return

    with open(txt_name,'a') as f:
        f.write('model: '+str(net)+'\n'+'time: '+time_str+'\n')

    for epoch in tqdm(range( start_epoch, epochs)):
        start_time=time.time()
        current_learning_rate=optimizer.state_dict()['param_groups'][0]['lr']
        tqdm.write('Current learning rate: '+str(current_learning_rate))
        with open(txt_name, 'a') as f:
            f.write('Current learning rate: ' + str(current_learning_rate) + '\n')

        train_acc,train_los=train(train_loader,model,criterion_cls,criterion_pt,optimizer,epoch+1)       #返回一个训练epoch平均精度和损失
        val_acc,val_los=validate(val_loader,model,criterion_cls,criterion_pt)        #返回一个验证epoch平均精度和损失
        scheduler.step()

        recorder.update(epoch,train_los,train_acc,val_los,val_acc)      #一个epoch记录精度和损失
        recorder.plot_curve(curve_name)      #绘制图形

        is_best=val_acc>best_acc
        best_acc=max(best_acc,val_acc)

        tqdm.write('Current best accuracy: '+str(best_acc.item()))

        with open(txt_name, 'a') as f:
            f.write('********************Current best accuracy: ' + str(best_acc.item()) + '\n')        #记录验证最高精度

        save_checkpoint({'epoch': epoch + 1,
                         'state_dict': model.state_dict(),
                         'best_acc': best_acc,
                         'optimizer': optimizer.state_dict(),
                         'recorder': recorder,}, is_best)

        end_time = time.time()
        epoch_time = end_time - start_time
        tqdm.write("An Epoch Time: "+ str(epoch_time))
        with open(txt_name, 'a') as f:      #写入一个epoch时间
            f.write('An epoch time: '+str(epoch_time) + '\n')


    # 训练
def train(train_loader,model,criterion_cls,criterion_pt,optimizer,epoch):
    losses = AverageMeter('Loss', ':.4f')
    top1 = AverageMeter('Accuracy', ':6.3f')
    progress = ProgressMeter(len(train_loader),     #传入多少个batch，损失和精度
                             [losses, top1],
                             prefix="Epoch: [{}]".format(epoch))

    model.train()
    #一个epoch
    for i,(images,targets) in enumerate(train_loader):
        targets=targets.to(device)
        images=images.to(device)
        out,heads=model(images)
        loss = alpha*criterion_cls(out,targets) +  (1-alpha)*criterion_pt(heads)   #89.3 89.4
        acc=accuracy(out,targets)
        losses.update(loss.item(),images.size(0))       #传入一个batch平均损失，batch_size，计算平均值
        top1.update(acc.item(),images.size(0))        #传入一个batch平均精度，batch_size，计算平均值

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if i% print_freq==0:        #显示训练精度和损失
            progress.display(i)
    
    return top1.avg, losses.avg     #返回一个epoch平均精度和损失

    # 验证
def validate(val_loader,model,criterion_cls,criterion_pt):
    losses = AverageMeter('Loss', ':.4f')
    top1 = AverageMeter('Accuracy', ':6.3f')
    progress = ProgressMeter(len(val_loader),
                             [losses, top1],
                             prefix='Test: ')
    model.eval()
    with torch.no_grad():
        for i,(images,targets) in enumerate(val_loader):

            targets=targets.to(device)
            images=images.to(device)

            out,heads=model(images)
            loss = criterion_cls(out,targets) + criterion_pt(heads)
            acc=accuracy(out,targets)
            losses.update(loss.item(),images.size(0))       #传入一个batch平均损失，batch_size，计算平均值
            top1.update(acc,images.size(0))        #传入一个batch平均精度，batch_size，计算平均值

            if i %  print_freq == 0:
                progress.display(i)
        tqdm.write(' **** Accuracy {top1.avg:.3f} *** '.format(top1=top1))
        # print(' **** Accuracy {top1.avg:.3f} *** '.format(top1=top1))       #输出验证平均精度
        with open(txt_name, 'a') as f:
            f.write(' * Accuracy {top1.avg:.3f}'.format(top1=top1) + '\n')      #写入验证平均精度

    return top1.avg,losses.avg
            
    #保存模型
def save_checkpoint(state, is_best):
    torch.save(state,  checkpoint_path)     #保存模型
    if is_best:     #若是最高精度，保存至最高精度模型
        shutil.copyfile( checkpoint_path,  best_checkpoint_path)

    #计算均值
class AverageMeter(object):     
    """Computes and stores the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

    #显示进度
class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)     #获取格式
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]       #
        entries += [str(meter) for meter in self.meters]
        print_txt = '\t'.join(entries)
        tqdm.write(print_txt)
        # print(print_txt)        #输出格式
        with open(txt_name, 'a') as f:      #写入格式
            f.write(print_txt + '\n')

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

    #返回一个batch精度
def accuracy(logits,labels):
    acc=(logits.argmax(dim=-1)==labels).float().mean()
    return acc*100.0

    #绘图
class RecorderMeter(object):
    """Computes and stores the minimum loss value and its epoch index"""

    def __init__(self, total_epoch):
        self.reset(total_epoch)

    def reset(self, total_epoch):
        self.total_epoch = total_epoch
        self.current_epoch = 0
        self.epoch_losses = np.zeros((self.total_epoch, 2), dtype=np.float32)    # [epoch, train/val]
        self.epoch_accuracy = np.zeros((self.total_epoch, 2), dtype=np.float32)  # [epoch, train/val]

    def update(self, idx, train_loss, train_acc, val_loss, val_acc):
        self.epoch_losses[idx, 0] = train_loss * 30
        self.epoch_losses[idx, 1] = val_loss * 30
        self.epoch_accuracy[idx, 0] = train_acc
        self.epoch_accuracy[idx, 1] = val_acc
        self.current_epoch = idx + 1

    def plot_curve(self, save_path):

        title = 'the accuracy/loss curve of train/val'
        dpi = 80
        width, height = 1800, 800
        legend_fontsize = 10
        figsize = width / float(dpi), height / float(dpi)

        fig = plt.figure(figsize=figsize)
        x_axis = np.array([i for i in range(self.total_epoch)])  # epochs
        y_axis = np.zeros(self.total_epoch)

        plt.xlim(0, self.total_epoch)
        plt.ylim(0, 100)
        interval_y = 5
        interval_x = 5
        plt.xticks(np.arange(0, self.total_epoch + interval_x, interval_x))
        plt.yticks(np.arange(0, 100 + interval_y, interval_y))
        plt.grid()
        plt.title(title, fontsize=20)
        plt.xlabel('the training epoch', fontsize=16)
        plt.ylabel('accuracy', fontsize=16)

        y_axis[:] = self.epoch_accuracy[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle='-', label='train-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_accuracy[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle='-', label='valid-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle=':', label='train-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle=':', label='valid-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        if save_path is not None:
            fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
            tqdm.write('Saved figure')
        plt.close(fig)

if __name__ == '__main__':
    main()


In [ ]:
%%writefile evaluation.py
import os
import torch
import torch.nn as nn
import torch.nn.parallel
import torch.backends.cudnn as cudnn
import torch.optim
import torch.utils.data
import torchvision.models as models
import torch.utils.data.distributed
import matplotlib.pyplot as plt
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import numpy as np
import datetime
from utils.loss import PartitionLoss
from train_rafdb import DrawConfusionMatrix
from network.my_model import *
from tqdm import tqdm

plt.switch_backend('agg')
now = datetime.datetime.now()
time_str = now.strftime("[%m-%d]-[%H-%M]-")


dataset_name='rafdb'

data_path               = 'data/'+dataset_name+'/test'
model_name              = dataset_name+'_'
checkpoint_path         = './checkpoint/' + model_name+time_str +'.pth'
best_checkpoint_path    = './checkpoint/'+model_name+time_str+'_best.pth'
txt_name                = './log/' + model_name +time_str+  '.txt'
curve_name              = './log/' + model_name +time_str+  '.png'
model_path              = 'experiment/'+dataset_name+'/'+dataset_name+'.pth'

device          = '2'
net             = 1
eval            = True
lr              = 0.01
momentum        = 0.9
weight_decay    = 1e-4
epochs          = 100
ls              = 15
batch_size      = 32
workers         = 4
print_freq      = 10
pretrained      =True


os.environ['KMP_DUPLICATE_LIB_OK']='True'
os.environ["CUDA_VISIBLE_DEVICES"] = device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def main():


    print('Training time: ' + now.strftime("%m-%d %H:%M"))
    model = Model_5(num_class=7,device=device)
    if dataset_name=='affectnet-8':
        model = Model_5(num_class=8,device=device)

    model=model.to(device)

    criterion_cls = nn.CrossEntropyLoss().to(device)
    criterion_pt = PartitionLoss()      # 最大化注意力方差

    cudnn.benchmark=True        #加速网络
    #if pretrained:

    #    checkpoint = torch.load(model_path,map_location=device)
    #    pre_trained_dict = checkpoint['state_dict']                                                          
    #    model.load_state_dict({k.replace('module.',''):v for k,v in pre_trained_dict.items()})
    if pretrained:
        tqdm.write('load pretrained model......')
        checkpoint = torch.load(model_path,map_location=torch.device('cpu'))
        pretrained_state_dict = checkpoint['state_dict']
        model_state_dict = model.state_dict()

        for key in pretrained_state_dict:
            if key in model_state_dict:
                model_state_dict[key] = pretrained_state_dict[key]

        model.load_state_dict(model_state_dict)
        tqdm.write('load succesful!')

    if not pretrained:
        tqdm.write('load pretrained model......')
        checkpoint = torch.load(model_path,map_location=torch.device('cpu'))
        pretrained_state_dict = checkpoint['state_dict']
        model_state_dict = model.state_dict()

        # for key in pretrained_state_dict:
        #     model_state_dict[key] = pretrained_state_dict[key]
        print(pretrained_state_dict)
        for key in model_state_dict:
            print(key)
            if key[:7]=='resnet2':
                print(key)
            else:
                # model_state_dict[key] = pretrained_state_dict['module.'+key]
                model_state_dict[key] = pretrained_state_dict[key]

        model.load_state_dict(model_state_dict)
        tqdm.write('load succesful!')
        state={'epoch': checkpoint['epoch'],
                         'state_dict': model.state_dict(),
                         'best_acc': checkpoint['best_acc'],
                         'optimizer': checkpoint['optimizer'],
                         'recorder': checkpoint['recorder'],}
        torch.save(state,  's.pth')
    val_dataset=datasets.ImageFolder(
        data_path,
        transforms.Compose([
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406],
                std=[0.229,0.224,0.225])
        ])
    )


    val_loader=torch.utils.data.DataLoader(
        val_dataset,
        batch_size= batch_size,
        shuffle=False,
        num_workers= workers,
        pin_memory=True
    )
 
    if  eval:
        validate(val_loader, model, criterion_cls,criterion_pt)
        return

    # 验证
def validate(val_loader,model,criterion_cls,criterion_pt):
    losses = AverageMeter('Loss', ':.4f')
    top1 = AverageMeter('Accuracy', ':6.3f')
    progress = ProgressMeter(len(val_loader),
                             [losses, top1],
                             prefix='Test: ')
    model.eval()
    labels_name=['Neutral', 'Happiness', 'Sadness', 'Surprise', 'Fear', 'Disgust', 'Anger']
    if dataset_name=='affectnet-8':
        labels_name=['Neutral', 'Happiness', 'Sadness', 'Surprise', 'Fear', 'Disgust', 'Anger','contempt']

    drawconfusionmatrix=DrawConfusionMatrix(labels_name=labels_name,path=dataset_name)
    with torch.no_grad():
        for i,(images,targets) in enumerate(val_loader):

            targets=targets.to(device)
            images=images.to(device)

            out,heads=model(images)
            loss = criterion_cls(out,targets) + criterion_pt(heads)

            predict_np=np.argmax(out.cpu().detach().numpy(),axis=-1)
            labels_np=targets.cpu().numpy()
            drawconfusionmatrix.update(predict_np,labels_np)

    
            acc=accuracy(out,targets)
            losses.update(loss.item(),images.size(0))       #传入一个batch平均损失，batch_size，计算平均值
            top1.update(acc,images.size(0))        #传入一个batch平均精度，batch_size，计算平均值

            if i %  print_freq == 0:
                progress.display(i)
        tqdm.write(' **** Accuracy {top1.avg:.3f} *** '.format(top1=top1))

        drawconfusionmatrix.drawMatrix()
        confusion_mat=drawconfusionmatrix.getMatrix()
        print(confusion_mat)

    return top1.avg,losses.avg
            

    #计算均值
class AverageMeter(object):     
    """Computes and stores the average and current value"""
    def __init__(self, name, fmt=':f'):
        self.name = name
        self.fmt = fmt
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

    def __str__(self):
        fmtstr = '{name} {val' + self.fmt + '} ({avg' + self.fmt + '})'
        return fmtstr.format(**self.__dict__)

    #显示进度
class ProgressMeter(object):
    def __init__(self, num_batches, meters, prefix=""):
        self.batch_fmtstr = self._get_batch_fmtstr(num_batches)     #获取格式
        self.meters = meters
        self.prefix = prefix

    def display(self, batch):
        entries = [self.prefix + self.batch_fmtstr.format(batch)]       #
        entries += [str(meter) for meter in self.meters]
        print_txt = '\t'.join(entries)
        tqdm.write(print_txt)

    def _get_batch_fmtstr(self, num_batches):
        num_digits = len(str(num_batches // 1))
        fmt = '{:' + str(num_digits) + 'd}'
        return '[' + fmt + '/' + fmt.format(num_batches) + ']'

    #返回一个batch精度
def accuracy(logits,labels):
    acc=(logits.argmax(dim=-1)==labels).float().mean()
    return acc*100.0

    #绘图
class RecorderMeter(object):
    """Computes and stores the minimum loss value and its epoch index"""

    def __init__(self, total_epoch):
        self.reset(total_epoch)

    def reset(self, total_epoch):
        self.total_epoch = total_epoch
        self.current_epoch = 0
        self.epoch_losses = np.zeros((self.total_epoch, 2), dtype=np.float32)    # [epoch, train/val]
        self.epoch_accuracy = np.zeros((self.total_epoch, 2), dtype=np.float32)  # [epoch, train/val]

    def update(self, idx, train_loss, train_acc, val_loss, val_acc):
        self.epoch_losses[idx, 0] = train_loss * 30
        self.epoch_losses[idx, 1] = val_loss * 30
        self.epoch_accuracy[idx, 0] = train_acc
        self.epoch_accuracy[idx, 1] = val_acc
        self.current_epoch = idx + 1

    def plot_curve(self, save_path):

        title = 'the accuracy/loss curve of train/val'
        dpi = 80
        width, height = 1800, 800
        legend_fontsize = 10
        figsize = width / float(dpi), height / float(dpi)

        fig = plt.figure(figsize=figsize)
        x_axis = np.array([i for i in range(self.total_epoch)])  # epochs
        y_axis = np.zeros(self.total_epoch)

        plt.xlim(0, self.total_epoch)
        plt.ylim(0, 100)
        interval_y = 5
        interval_x = 5
        plt.xticks(np.arange(0, self.total_epoch + interval_x, interval_x))
        plt.yticks(np.arange(0, 100 + interval_y, interval_y))
        plt.grid()
        plt.title(title, fontsize=20)
        plt.xlabel('the training epoch', fontsize=16)
        plt.ylabel('accuracy', fontsize=16)

        y_axis[:] = self.epoch_accuracy[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle='-', label='train-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_accuracy[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle='-', label='valid-accuracy', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 0]
        plt.plot(x_axis, y_axis, color='g', linestyle=':', label='train-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        y_axis[:] = self.epoch_losses[:, 1]
        plt.plot(x_axis, y_axis, color='y', linestyle=':', label='valid-loss-x30', lw=2)
        plt.legend(loc=4, fontsize=legend_fontsize)

        if save_path is not None:
            fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
            tqdm.write('Saved figure')
        plt.close(fig)

if __name__ == '__main__':
    main()



In [ ]:
# Install required dependencies
!pip install pandas matplotlib tqdm torchvision

In [ ]:
# Run Training on RAF-DB
!python train_rafdb.py

In [ ]:
import glob
import shutil
# Find the best checkpoint and set it up for evaluation.py
best_ckpt = glob.glob('checkpoint/*_best.pth')
if best_ckpt:
    print(f'Found best checkpoint: {best_ckpt[0]}')
    shutil.copy(best_ckpt[0], 'experiment/rafdb/rafdb.pth')
else:
    print('No checkpoint found!')


In [ ]:
# Run Evaluation on RAF-DB
!python evaluation.py

In [ ]:
from IPython.display import Image, display
import glob

# Display Training Curves
curve_images = glob.glob('log/*.png')
if curve_images:
    print('Training & Validation Curves:')
    display(Image(filename=curve_images[-1]))

# Display Confusion Matrix
cm_images = glob.glob('experiment/visual/confusion_matrix/*.png')
if cm_images:
    print('\nConfusion Matrix:')
    display(Image(filename=cm_images[-1]))
